# Near-Earth Object Data Engineering

This notebook prepares and combines two asteroid datasets for the machine learning classification.

The main tasks completed are:

1. Loading and inspecting both of the datasets
2. Checking the data quality
3. Validating the join columns
4. Investigating unmatched records
5. Combining the datasets
6. Saving the processed data

In [2]:
# pandas is used throughout for loading, checking and combining the datasets
import pandas as pd


In [3]:
# Load the main NEO dataset that will be used for the classification
primary = pd.read_csv("nearest-earth-objects(1910-2024).csv")

# Load the larger supplementary dataset containing extra orbital information
# low_memory=False stops pandas inferring data types in smaller chunks, which can cause mixed-type warnings on large files
supplementary = pd.read_csv(
    "dataset.csv",
    low_memory=False
)


In [4]:
# Check the size of both datasets before doing any cleaning or joining
# This provides a useful baseline for checking whether rows are unexpectedly lost later
print("Primary dataset shape:", primary.shape)
print("Supplementary dataset shape:", supplementary.shape)


Primary dataset shape: (338199, 9)
Supplementary dataset shape: (958524, 45)


In [5]:
# Preview the first few rows of the primary dataset to check the columns and values loaded as expected
primary.head()


,neo_id,name,absolute_magnitude,estimated_diameter_min,estimated_diameter_max,orbiting_body,relative_velocity,miss_distance,is_hazardous
0,2162117,162117 (1998 SD15),19.14,0.394962,0.883161,Earth,71745.401048,5.814362e+07,False
1,2349507,349507 (2008 QY),18.50,0.530341,1.185878,Earth,109949.757148,5.580105e+07,True
2,2455415,455415 (2003 GA),21.45,0.136319,0.304818,Earth,24865.506798,6.720689e+07,False
3,3132126,(2002 PB),20.63,0.198863,0.444672,Earth,78890.076805,3.039644e+07,False
4,3557844,(2011 DW),22.70,0.076658,0.171412,Earth,56036.519484,6.311863e+07,False


In [6]:
# Do the same for the supplementary dataset, which contains considerably more columns
supplementary.head()


,id,spkid,full_name,pdes,name,prefix,neo,pha,H,diameter,...,sigma_i,sigma_om,sigma_w,sigma_ma,sigma_ad,sigma_n,sigma_tp,sigma_per,class,rms
0,a0000001,2000001,1 Ceres,1,Ceres,NaN,N,N,3.40,939.400,...,4.608900e-09,6.168800e-08,6.624800e-08,7.820700e-09,1.111300e-11,1.196500e-12,3.782900e-08,9.415900e-09,MBA,0.43301
1,a0000002,2000002,2 Pallas,2,Pallas,NaN,N,N,4.20,545.000,...,3.469400e-06,6.272400e-06,9.128200e-06,8.859100e-06,4.961300e-09,4.653600e-10,4.078700e-05,3.680700e-06,MBA,0.35936
2,a0000003,2000003,3 Juno,3,Juno,NaN,N,N,5.33,246.596,...,3.223100e-06,1.664600e-05,1.772100e-05,8.110400e-06,4.363900e-09,4.413400e-10,3.528800e-05,3.107200e-06,MBA,0.33848
3,a0000004,2000004,4 Vesta,4,Vesta,NaN,N,N,3.00,525.400,...,2.170600e-07,3.880800e-07,1.789300e-07,1.206800e-06,1.648600e-09,2.612500e-10,4.103700e-06,1.274900e-06,MBA,0.39980
4,a0000005,2000005,5 Astraea,5,Astraea,NaN,N,N,6.90,106.699,...,2.740800e-06,2.894900e-05,2.984200e-05,8.303800e-06,4.729000e-09,5.522700e-10,3.474300e-05,3.490500e-06,MBA,0.52191


## Data quality checks

The datasets were inspected for missing values, duplicated rows and unsuitable data types before integration.

In [7]:
# Inspect the primary dataset structure, including column names, data types and non-null counts
primary.info()


<class 'pandas.DataFrame'>
RangeIndex: 338199 entries, 0 to 338198
Data columns (total 9 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   neo_id                  338199 non-null  int64  
 1   name                    338199 non-null  str    
 2   absolute_magnitude      338171 non-null  float64
 3   estimated_diameter_min  338171 non-null  float64
 4   estimated_diameter_max  338171 non-null  float64
 5   orbiting_body           338199 non-null  str    
 6   relative_velocity       338199 non-null  float64
 7   miss_distance           338199 non-null  float64
 8   is_hazardous            338199 non-null  bool   
dtypes: bool(1), float64(5), int64(1), str(2)
memory usage: 21.0 MB


In [8]:
# Count missing values in each primary column to identify exactly which fields need cleaning
primary.isnull().sum()


neo_id                     0
name                       0
absolute_magnitude        28
estimated_diameter_min    28
estimated_diameter_max    28
orbiting_body              0
relative_velocity          0
miss_distance              0
is_hazardous               0
dtype: int64

In [9]:
# Check for completely duplicated rows in the primary dataset
primary.duplicated().sum()


np.int64(0)

In [13]:
# Only a very small number of primary rows contain missing values, so remove these before modelling
# .copy() makes primary_clean a separate dataframe rather than a view of the original data
primary_clean = primary.dropna().copy()

# Compare row counts before and after cleaning to make the amount of removed data clear
print("Rows before cleaning:", len(primary))
print("Rows after cleaning:", len(primary_clean))
print("Rows removed:", len(primary) - len(primary_clean))


Rows before cleaning: 338199
Rows after cleaning: 338171
Rows removed: 28


In [10]:
# Inspect the supplementary dataset in the same way, especially because it has many more columns
supplementary.info()


<class 'pandas.DataFrame'>
RangeIndex: 958524 entries, 0 to 958523
Data columns (total 45 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   id              958524 non-null  str    
 1   spkid           958524 non-null  int64  
 2   full_name       958524 non-null  str    
 3   pdes            958524 non-null  str    
 4   name            22064 non-null   str    
 5   prefix          18 non-null      str    
 6   neo             958520 non-null  str    
 7   pha             938603 non-null  str    
 8   H               952261 non-null  float64
 9   diameter        136209 non-null  float64
 10  albedo          135103 non-null  float64
 11  diameter_sigma  136081 non-null  float64
 12  orbit_id        958524 non-null  str    
 13  epoch           958524 non-null  float64
 14  epoch_mjd       958524 non-null  int64  
 15  epoch_cal       958524 non-null  float64
 16  equinox         958524 non-null  str    
 17  e               95852

In [11]:
# Check missing values across all supplementary fields
# Some columns are very incomplete, so this helps decide which ones are actually suitable to carry forward
supplementary.isnull().sum()


id                     0
spkid                  0
full_name              0
pdes                   0
name              936460
prefix            958506
neo                    4
pha                19921
H                   6263
diameter          822315
albedo            823421
diameter_sigma    822443
orbit_id               0
epoch                  0
epoch_mjd              0
epoch_cal              0
equinox                0
e                      0
a                      0
q                      0
i                      0
om                     0
w                      0
ma                     1
ad                     4
n                      0
tp                     0
tp_cal                 0
per                    4
per_y                  1
moid               19921
moid_ld              127
sigma_e            19922
sigma_a            19922
sigma_q            19922
sigma_i            19922
sigma_om           19922
sigma_w            19922
sigma_ma           19922
sigma_ad           19926


In [12]:
# Check whether the supplementary dataset contains any completely duplicated rows
supplementary.duplicated().sum()


np.int64(0)

In [14]:
# Focus the next checks on the identifier, hazard information and physical/orbital fields that are most relevant
# This is easier to interpret than repeatedly checking all 45 supplementary columns
columns_to_check = [
    "spkid",
    "full_name",
    "neo",
    "pha",
    "H",
    "diameter",
    "albedo",
    "e",
    "a",
    "q",
    "i",
    "per",
    "moid",
    "class"
]

# Check how complete these potentially useful columns are
supplementary[columns_to_check].isnull().sum()


spkid             0
full_name         0
neo               4
pha           19921
H              6263
diameter     822315
albedo       823421
e                 0
a                 0
q                 0
i                 0
per               4
moid          19921
class             0
dtype: int64

In [15]:
# Check the data types of the selected supplementary columns before attempting to join or model with them
supplementary[columns_to_check].dtypes


spkid          int64
full_name        str
neo              str
pha              str
H            float64
diameter     float64
albedo       float64
e            float64
a            float64
q            float64
i            float64
per          float64
moid         float64
class            str
dtype: object

In [16]:
# spkid will be used as the supplementary join key, so first check that the same ID is not repeated
supplementary["spkid"].duplicated().sum()


np.int64(0)

In [17]:
# Count the number of unique spkid values as a second check that the join key is unique
supplementary["spkid"].nunique()


958524

In [18]:
# Compare the total row count with the unique spkid count above
# If they are the same, each supplementary record has its own spkid and the join should not duplicate primary rows
len(supplementary)


958524

In [19]:
# Check which cleaned primary NEO IDs are actually present in the supplementary dataset
# Doing this before the join shows how much primary data would be lost if an inner join were used
matches = primary_clean["neo_id"].isin(supplementary["spkid"])

# Report both the number of matched/unmatched records and the overall match percentage
print("Matched rows:", matches.sum())
print("Unmatched rows:", (~matches).sum())
print("Match rate:", round(matches.mean() * 100, 2), "%")


Matched rows: 241900
Unmatched rows: 96271
Match rate: 71.53 %


In [20]:
# Split the primary data into matched and unmatched groups to investigate whether the missing matches are random
matched = primary_clean[matches]
unmatched = primary_clean[~matches]

# Compare the hazardous-object rate in each group
# A noticeable difference would mean dropping unmatched rows could change the class distribution and introduce selection bias
print(
    "Matched hazardous rate:",
    round(matched["is_hazardous"].mean() * 100, 2),
    "%"
)

print(
    "Unmatched hazardous rate:",
    round(unmatched["is_hazardous"].mean() * 100, 2),
    "%"
)


Matched hazardous rate: 15.35 %
Unmatched hazardous rate: 6.28 %


In [21]:
# Keep only the supplementary orbital fields needed for the combined dataset rather than carrying all 45 columns forward
# spkid is retained because it is required for the join and can also be used to confirm whether a match was found
supplementary_selected = supplementary[
    [
        "spkid",
        "e",
        "a",
        "q",
        "i",
        "per",
        "class"
    ]
].copy()


In [22]:
# Join the supplementary orbital features onto the cleaned primary data using the object IDs
# A left join keeps every primary record, including the NEOs that did not have a supplementary match
combined = primary_clean.merge(
    supplementary_selected,
    left_on="neo_id",
    right_on="spkid",
    how="left"
)


In [23]:
# Validate the join by checking that the number of rows has not changed
# Because spkid is unique, the left join should add columns without creating duplicate primary records
print("Primary rows before join:", len(primary_clean))
print("Combined rows after join:", len(combined))


Primary rows before join: 338171
Combined rows after join: 338171


In [24]:
# Preview the combined dataframe to confirm the supplementary fields have been added correctly
combined.head()


,neo_id,name,absolute_magnitude,estimated_diameter_min,estimated_diameter_max,orbiting_body,relative_velocity,miss_distance,is_hazardous,spkid,e,a,q,i,per,class
0,2162117,162117 (1998 SD15),19.14,0.394962,0.883161,Earth,71745.401048,5.814362e+07,False,2162117.0,0.344834,0.932486,0.610933,26.795548,328.898516,ATE
1,2349507,349507 (2008 QY),18.50,0.530341,1.185878,Earth,109949.757148,5.580105e+07,True,2349507.0,0.581509,1.167263,0.488489,13.576888,460.629169,APO
2,2455415,455415 (2003 GA),21.45,0.136319,0.304818,Earth,24865.506798,6.720689e+07,False,2455415.0,0.191237,1.281544,1.036465,3.842210,529.905950,AMO
3,3132126,(2002 PB),20.63,0.198863,0.444672,Earth,78890.076805,3.039644e+07,False,3132126.0,0.343319,1.072262,0.704134,32.951327,405.554991,APO
4,3557844,(2011 DW),22.70,0.076658,0.171412,Earth,56036.519484,6.311863e+07,False,3557844.0,0.275227,0.839730,0.608614,25.066956,281.065906,ATE


In [25]:
# Create an explicit flag showing whether each primary record found a supplementary match
# This is clearer to work with later than repeatedly checking whether spkid is null
combined["matched_supplementary"] = combined["spkid"].notnull()


In [26]:
# Count matched and unmatched rows again after the join to make sure the result agrees with the earlier validation
combined["matched_supplementary"].value_counts()


matched_supplementary
True     241900
False     96271
Name: count, dtype: int64

In [27]:
# Final missing-value check on the combined dataset
# Nulls in the supplementary fields are expected for records where no match was available, while the primary fields should remain complete
combined.isnull().sum()


neo_id                        0
name                          0
absolute_magnitude            0
estimated_diameter_min        0
estimated_diameter_max        0
orbiting_body                 0
relative_velocity             0
miss_distance                 0
is_hazardous                  0
spkid                     96271
e                         96271
a                         96271
q                         96271
i                         96271
per                       96271
class                     96271
matched_supplementary         0
dtype: int64

In [28]:
# Save the processed dataset for the later analysis and machine learning notebooks
# index=False prevents pandas adding the dataframe row index as an unnecessary CSV column
combined.to_csv("combined_neo_dataset.csv", index=False)


### Join outcome

A left join was used to retain all cleaned primary records.  
241,900 records matched the supplementary dataset, while 96,271 remained unmatched.  
This avoided removing nearly 30% of the primary data and reduced the risk of introducing selection bias.